# Single STAC Item: `open_item` and `open_cog`

Two single-item helpers in `lazycogs`, both lazy and read at the asset's native grid:

- `lazycogs.open_item` — stack several same-grid bands of one STAC item into a `(band, y, x)` DataArray.
- `lazycogs.open_cog` — read one Cloud-Optimized GeoTIFF at its native CRS, resolution, and shape.

First, find one low-cloud Sentinel-2 scene and configure a store for the public bucket (no credentials needed).

In [ ]:
import rustac
from obstore.store import S3Store

import lazycogs

items = await rustac.search(
    href="https://earth-search.aws.element84.com/v1",
    collections=["sentinel-2-c1-l2a"],
    bbox=[4.8, 52.3, 5.0, 52.5],  # Amsterdam
    datetime="2023-06-01/2023-06-30",
    query={"eo:cloud_cover": {"lt": 10}},
    limit=1,
)
item = items[0]

store = S3Store(
    bucket="e84-earth-search-sentinel-data",
    region="us-west-2",
    skip_signature=True,
    virtual_hosted_style_request=True,
)

## Open a Multi-Band Item with `open_item`

`lazycogs.open_item` stacks several single-band assets of one STAC item into a `(band, y, x)` DataArray at their shared native grid. Here we request the `red` and `green` bands. Notice that both of these assets have the same grid. If we were to pick bands with different grids, `lazycogs.open_item` would raise an error, `lazycogs.open` would then be the correct function to use.

Opening and selecting read headers only. Select the band and window before loading raw, full-resolution pixels.

In [ ]:
da = lazycogs.open_item(item, bands=["red", "green"], store=store)
window = da.sel(band="red").isel(y=slice(100, 356), x=slice(200, 456))
pixels = window.load()
pixels

## Open a Single COG with `open_cog`

`lazycogs.open_cog` reads one Cloud-Optimized GeoTIFF in place — native CRS, resolution, and shape, with no reprojection. Here we open just the `nir` asset of the same item directly from its href.

In [ ]:
nir = lazycogs.open_cog(item["assets"]["nir"]["href"], store=store)
nir_window = nir.isel(y=slice(100, 356), x=slice(200, 456))
nir_window.load()